# LSS Step Through

这个 notebook 用来逐行跑通 `learn2027/src` 里的 LSS 源码。

学习目标不是一口气训练模型，而是看清楚每一步：

```text
数据进来是什么 shape
代码做了什么变换
为什么 LSS 需要这个变换
变换后数据变成什么样子
```

In [ ]:
from pathlib import Path
import os
import sys

def is_repo_root(path):
    return (path / "learn2027" / "src" / "models.py").exists()

def find_repo_root():
    # 1. Best case: Jupyter was started from the repo or a child directory.
    for path in [Path.cwd(), *Path.cwd().parents]:
        if is_repo_root(path):
            return path

    # 2. Environment variable fallback, useful when VS Code starts notebooks
    #    from a directory outside this repository.
    env_repo_root = os.environ.get("LSS_REPO_ROOT")
    if env_repo_root:
        repo_root = Path(env_repo_root).expanduser()
        if is_repo_root(repo_root):
            return repo_root

    # 3. Optional local file config. The real config file should stay outside
    #    Git or be ignored by Git.
    try:
        import importlib.util
        config_candidates = []
        if os.environ.get("LSS_LOCAL_CONFIG"):
            config_candidates.append(Path(os.environ["LSS_LOCAL_CONFIG"]).expanduser())
        config_candidates.append(Path.home() / ".config" / "lss2027" / "local_config.py")
        for path in [Path.cwd(), *Path.cwd().parents]:
            config_candidates.append(path / "learn2027" / "local_config.py")
        for config_path in config_candidates:
            if not config_path.exists():
                continue
            spec = importlib.util.spec_from_file_location("lss_local_config", config_path)
            config = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(config)
            repo_root = Path(config.REPO_ROOT).expanduser()
            if is_repo_root(repo_root):
                return repo_root
    except Exception as exc:
        print("Ignoring invalid local_config.py:", exc)

    raise FileNotFoundError(
        "Could not find the lift-splat-shoot repo root. "
        "Start Jupyter from the repo root, set LSS_REPO_ROOT, set "
        "LSS_LOCAL_CONFIG, or create ~/.config/lss2027/local_config.py."
    )

repo_root = find_repo_root()
learn_root = repo_root / "learn2027"
mpl_config = learn_root / ".matplotlib"
mpl_config.mkdir(parents=True, exist_ok=True)
os.environ["MPLCONFIGDIR"] = str(mpl_config)
sys.path.insert(0, str(learn_root))

print("repo_root:", repo_root)
print("using learning src:", learn_root / "src")


In [ ]:
import torch
from io import BytesIO

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

import matplotlib.pyplot as plt
from IPython.display import display, Image as DisplayImage

from src.models import LiftSplatShoot


def _display_fig(fig):
    """Display exactly one PNG for a Matplotlib figure in notebook outputs."""
    buf = BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=120)
    plt.close(fig)
    display(DisplayImage(data=buf.getvalue()))

torch.set_printoptions(precision=3, sci_mode=False)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())


## 1. 建立和源码一致的配置

这些配置来自 `src/train.py`。先不要改它们，先看懂默认设置。

In [ ]:
grid_conf = {
    "xbound": [-50.0, 50.0, 0.5],
    "ybound": [-50.0, 50.0, 0.5],
    "zbound": [-10.0, 10.0, 20.0],
    "dbound": [4.0, 45.0, 1.0],
}

data_aug_conf = {
    "resize_lim": (0.193, 0.225),
    "final_dim": (128, 352),
    "rot_lim": (-5.4, 5.4),
    "H": 900,
    "W": 1600,
    "rand_flip": True,
    "bot_pct_lim": (0.0, 0.22),
    "cams": [
        "CAM_FRONT_LEFT", "CAM_FRONT", "CAM_FRONT_RIGHT",
        "CAM_BACK_LEFT", "CAM_BACK", "CAM_BACK_RIGHT",
    ],
    "Ncams": 5,
}

print(grid_conf)
print(data_aug_conf)


## 2. 先只实例化模型

`LiftSplatShoot.__init__` 会创建：

```text
dx / bx / nx
frustum
CamEncode
BevEncode
```

第一次运行这里可能会下载 EfficientNet 预训练权重。

In [ ]:
model = LiftSplatShoot(grid_conf, data_aug_conf, outC=1)
model.eval()

print("dx:", model.dx)
print("bx:", model.bx)
print("nx:", model.nx)
print("frustum shape:", tuple(model.frustum.shape))
print("D:", model.D)
print("camC:", model.camC)


## 3. 可视化 `create_frustum()` 的结果

`frustum[d, h, w] = [image_x, image_y, depth]`

它的目的：给每个图像特征点准备多个候选深度。

In [ ]:
frustum = model.frustum.detach()
print("frustum shape:", tuple(frustum.shape))
print("first point:", frustum[0, 0, 0])
print("last point:", frustum[-1, -1, -1])

D, fH, fW, _ = frustum.shape
depth_values = frustum[:, 0, 0, 2]
print("depth values:", depth_values[:10], "...", depth_values[-5:])

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(depth_values.numpy(), marker="o")
ax.set_title("Depth candidates used by create_frustum")
ax.set_xlabel("depth index")
ax.set_ylabel("meters")
ax.grid(True)
_display_fig(fig)


In [ ]:
d_index = 0
xy = frustum[d_index, :, :, :2].reshape(-1, 2)

fig, ax = plt.subplots(figsize=(8, 3))
ax.scatter(xy[:, 0], xy[:, 1], s=30)
ax.set_title(f"Image-plane feature locations at depth={frustum[d_index,0,0,2].item():.1f}m")
ax.set_xlim(0, data_aug_conf["final_dim"][1])
ax.set_ylim(data_aug_conf["final_dim"][0], 0)
ax.set_xlabel("image x")
ax.set_ylabel("image y")
ax.grid(True)
_display_fig(fig)


## 4. 下一步

下一节建议进入：

```python
CamEncode.get_depth_feat()
```

先观察为什么 `depthnet` 输出 `D + C` 个通道。

## 5. 真正的一步一个 cell：forward 源码展开

从这里开始，每个 code cell 只做一个主要 tensor 变换，然后马上显示该步骤的 shape、统计量和可视化。

执行方式：从上往下顺序运行。


In [ ]:
import math


def _tensor_stats(tensor):
    if not torch.is_tensor(tensor):
        return "not a tensor"
    data = tensor.detach()
    msg = f"shape={tuple(data.shape)} dtype={data.dtype}"
    if data.numel() and torch.is_floating_point(data):
        flat = data.float().reshape(-1)
        msg += (
            f" min={flat.min().item():.4g}"
            f" max={flat.max().item():.4g}"
            f" mean={flat.mean().item():.4g}"
            f" std={flat.std(unbiased=False).item():.4g}"
        )
    return msg


def _sample_rows(rows, max_points=5000):
    if rows.shape[0] <= max_points:
        return rows
    idx = torch.linspace(0, rows.shape[0] - 1, max_points).long()
    return rows[idx]


def _as_heatmap(tensor):
    data = tensor.detach().float().cpu()
    while data.ndim > 4:
        data = data[0]
    if data.ndim == 4:
        data = data[0]
    if data.ndim == 3:
        if data.shape[0] <= 256:
            data = data.mean(0)
        else:
            data = data[..., 0]
    if data.ndim == 2:
        return data
    if data.ndim == 1:
        return data.view(1, -1)
    if data.ndim == 0:
        return data.view(1, 1)
    return data.reshape(data.shape[0], -1)


def show_heatmap(name, tensor):
    print(name)
    print(_tensor_stats(tensor))
    heat = _as_heatmap(tensor)
    fig, ax = plt.subplots(figsize=(7, 3))
    im = ax.imshow(heat, cmap="viridis", aspect="auto")
    fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    ax.set_title(name)
    _display_fig(fig)


def show_points2d(name, tensor):
    print(name)
    print(_tensor_stats(tensor))
    pts = tensor.detach().float().cpu().reshape(-1, tensor.shape[-1])
    pts = _sample_rows(pts)
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(pts[:, 0], pts[:, 1], s=1, alpha=0.35)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(name)
    ax.axis("equal")
    _display_fig(fig)


def show_points3d(name, tensor):
    print(name)
    print(_tensor_stats(tensor))
    pts = tensor.detach().float().cpu().reshape(-1, tensor.shape[-1])
    pts = _sample_rows(pts)
    fig = plt.figure(figsize=(6, 5))
    ax = fig.add_subplot(111, projection="3d")
    color = pts[:, 2] if pts.shape[1] >= 3 else None
    ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], c=color, s=1, alpha=0.35, cmap="viridis")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")
    ax.set_title(name)
    _display_fig(fig)


def show_hist(name, tensor, bins=60):
    print(name)
    print(_tensor_stats(tensor))
    flat = tensor.detach().float().cpu().reshape(-1, 1)
    flat = _sample_rows(flat).reshape(-1)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.hist(flat.numpy(), bins=bins)
    ax.set_title(name)
    _display_fig(fig)


def show_mask(name, mask):
    print(name)
    print(_tensor_stats(mask))
    flat = mask.detach().cpu().reshape(-1).bool()
    kept = int(flat.sum())
    dropped = int(flat.numel() - kept)
    fig, ax = plt.subplots(figsize=(4, 3))
    ax.bar(["kept", "dropped"], [kept, dropped])
    ax.set_title(name)
    _display_fig(fig)

print("visual helpers ready")


## 6. 输入准备：同一个 BEV 场景贯穿后续所有步骤

这一节准备一个“不依赖 nuScenes、但几何一致”的输入。

后面的 `get_geometry()`、`voxel_pooling()`、`model.forward()` 都会继续使用这里创建的同一批变量：

```text
imgs, rots, trans, intrins, post_rots, post_trans
```

为了保证第 6 部分的相机图、BEV 图和后续 Geometry 完全一致，这里先关闭图像增强修正：

```text
post_rots = identity
post_trans = 0
```

所以从第 6 步往后，所有坐标都在同一套相机内参/外参下流动。等这条主线看懂后，再单独观察 `post_rots/post_trans` 如何影响 frustum。


In [ ]:
B, N, C, imH, imW = 1, 6, 3, data_aug_conf["final_dim"][0], data_aug_conf["final_dim"][1]
camera_names = [
    "front_left", "front", "front_right",
    "back_left", "back", "back_right",
]

# 1. 先定义同一个 ego/BEV 世界场景。
#    之后 6 个相机图都只从这个场景投影得到，保证和 BEV 图一致。
world_objects = [
    {"name": "blue_car_front", "xy": (16.0, -1.4), "size": (3.8, 1.8, 1.5), "color": (0.10, 0.45, 1.00)},
    {"name": "red_car_left", "xy": (8.5, 4.5), "size": (3.6, 1.7, 1.5), "color": (1.00, 0.22, 0.15)},
    {"name": "yellow_car_right", "xy": (4.0, -6.0), "size": (3.8, 1.8, 1.5), "color": (1.00, 0.78, 0.12)},
    {"name": "green_car_back", "xy": (-10.0, 1.6), "size": (3.8, 1.8, 1.5), "color": (0.10, 0.75, 0.35)},
]
world_cones = [
    (7.0, -2.8), (10.0, -2.8), (13.0, -2.8),
    (-5.0, 3.2), (-8.0, 3.2),
]

# 2. 构造相机内参。
fx = fy = 220.0
cx = (imW - 1) / 2.0
cy = (imH - 1) / 2.0
intrins = torch.eye(3).view(1, 1, 3, 3).repeat(B, N, 1, 1)
intrins[:, :, 0, 0] = fx
intrins[:, :, 1, 1] = fy
intrins[:, :, 0, 2] = cx
intrins[:, :, 1, 2] = cy

# 3. 构造 6 个相机外参。
#    camera 坐标约定：x 向右，y 向下，z 向前。
#    ego 坐标约定：x 向前，y 向左，z 向上。
def yaw_rot(yaw):
    c = torch.cos(torch.tensor(yaw))
    s = torch.sin(torch.tensor(yaw))
    return torch.tensor([
        [c, -s, 0.0],
        [s,  c, 0.0],
        [0.0, 0.0, 1.0],
    ])

front_cam_to_ego = torch.tensor([
    [0.0,  0.0, 1.0],
    [-1.0, 0.0, 0.0],
    [0.0, -1.0, 0.0],
])
yaws = torch.tensor([math.pi / 4, 0.0, -math.pi / 4, 3 * math.pi / 4, math.pi, -3 * math.pi / 4])
rots = torch.stack([yaw_rot(float(yaw)).matmul(front_cam_to_ego) for yaw in yaws])
rots = rots.view(1, N, 3, 3).repeat(B, 1, 1, 1)
trans = torch.tensor([
    [1.4,  0.8, 1.5],
    [1.7,  0.0, 1.5],
    [1.4, -0.8, 1.5],
    [-1.2,  0.8, 1.5],
    [-1.5,  0.0, 1.5],
    [-1.2, -0.8, 1.5],
]).view(1, N, 3).repeat(B, 1, 1)

# 4. 关闭 post transform，保证第 6 部分的图和后续 Geometry 使用同一套像素坐标。
#    真实训练中 post_rots/post_trans 来自 resize/crop/flip/rotate；
#    这里先不引入增强，避免教学样例前后不一致。
post_rots = torch.eye(3).view(1, 1, 3, 3).repeat(B, N, 1, 1)
post_trans = torch.zeros(B, N, 3)

# 5. 投影工具：ego 3D 点 -> 第 cam_i 个相机的图像像素。
def project_ego_to_image(points_ego, cam_i):
    pts = torch.as_tensor(points_ego, dtype=torch.float32)
    pts_cam = rots[0, cam_i].T.matmul((pts - trans[0, cam_i]).T).T
    pts_img = intrins[0, cam_i].matmul(pts_cam.T).T
    uv = pts_img[:, :2] / pts_img[:, 2:3].clamp(min=1e-6)
    visible = (pts_cam[:, 2] > 0.2)
    visible &= (uv[:, 0] >= 0) & (uv[:, 0] < imW)
    visible &= (uv[:, 1] >= 0) & (uv[:, 1] < imH)
    return uv, pts_cam[:, 2], visible


def paint_rect(img_hwc, u, v, radius, color):
    u0 = max(0, int(round(float(u))) - radius)
    u1 = min(imW, int(round(float(u))) + radius + 1)
    v0 = max(0, int(round(float(v))) - radius)
    v1 = min(imH, int(round(float(v))) + radius + 1)
    if u0 < u1 and v0 < v1:
        img_hwc[v0:v1, u0:u1] = torch.tensor(color, dtype=img_hwc.dtype)


def box_corners(cx_, cy_, length, width, z):
    return torch.tensor([
        [cx_ - length / 2, cy_ - width / 2, z],
        [cx_ + length / 2, cy_ - width / 2, z],
        [cx_ + length / 2, cy_ + width / 2, z],
        [cx_ - length / 2, cy_ + width / 2, z],
    ], dtype=torch.float32)

# 6. 严格从同一个 BEV 场景投影生成 6 张相机示意图。
#    这些图是几何一致的 schematic，不是假装成真实照片。
imgs = torch.zeros(B, N, C, imH, imW)
yy_img, xx_img = torch.meshgrid(torch.linspace(0, 1, imH), torch.linspace(0, 1, imW), indexing="ij")
for cam_i in range(N):
    horizon = 0.38
    sky = torch.stack([0.50 - 0.12 * yy_img, 0.68 - 0.10 * yy_img, 0.92 - 0.05 * yy_img], dim=2)
    ground = torch.stack([0.25 + 0.08 * yy_img, 0.30 + 0.10 * yy_img, 0.27 + 0.05 * yy_img], dim=2)
    img_hwc = torch.where((yy_img < horizon).unsqueeze(2), sky, ground).clamp(0, 1)

    # 车道线：从同一组 BEV 线段采样后投影。
    for y_lane in [-3.5, 0.0, 3.5]:
        xs = torch.linspace(-24, 30, 220)
        lane_pts = torch.stack([xs, torch.full_like(xs, y_lane), torch.zeros_like(xs)], dim=1)
        uv, depth, visible = project_ego_to_image(lane_pts, cam_i)
        for (u, v), d, ok in zip(uv, depth, visible):
            if bool(ok):
                color = (1.0, 1.0, 1.0) if y_lane != 0.0 else (1.0, 0.92, 0.55)
                radius = 1 if float(d) > 8 else 2
                paint_rect(img_hwc, u, v, radius, color)

    # 车辆：投影同一批 BEV 3D box 的角点和中心。
    for obj in world_objects:
        cx_obj, cy_obj = obj["xy"]
        length, width, height = obj["size"]
        corners = torch.cat([
            box_corners(cx_obj, cy_obj, length, width, 0.0),
            box_corners(cx_obj, cy_obj, length, width, height),
            torch.tensor([[cx_obj, cy_obj, height * 0.5]], dtype=torch.float32),
        ], dim=0)
        uv, depth, visible = project_ego_to_image(corners, cam_i)
        if bool(visible.any()):
            center_uv = uv[visible].mean(0)
            center_depth = depth[visible].mean()
            radius = max(2, min(12, int(50 / float(center_depth))))
            paint_rect(img_hwc, center_uv[0], center_uv[1], radius, obj["color"])

    # 路锥：投影同一批 BEV 点。
    for cone_x, cone_y in world_cones:
        cone_pts = torch.tensor([
            [cone_x, cone_y, 0.0],
            [cone_x, cone_y, 0.8],
        ])
        uv, depth, visible = project_ego_to_image(cone_pts, cam_i)
        if bool(visible.any()):
            center_uv = uv[visible].mean(0)
            radius = max(1, min(5, int(25 / float(depth[visible].mean()))))
            paint_rect(img_hwc, center_uv[0], center_uv[1], radius, (1.0, 0.48, 0.05))

    imgs[0, cam_i] = img_hwc.permute(2, 0, 1)

fig, axes = plt.subplots(2, 3, figsize=(10, 4.8))
for cam_i, ax in enumerate(axes.reshape(-1)):
    ax.imshow(imgs[0, cam_i].permute(1, 2, 0).detach().cpu().numpy())
    ax.set_title(camera_names[cam_i])
    ax.axis("off")
fig.suptitle("00a shared BEV scene projected into 6 schematic camera images")
_display_fig(fig)

print("imgs:", tuple(imgs.shape))
print("intrins[front]:\\n", intrins[0, 1])
print("rots[front]:\\n", rots[0, 1])
print("trans:", trans[0])
print("post_rots[front]:\\n", post_rots[0, 1])
print("post_trans[front]:", post_trans[0, 1])


In [ ]:
# 00b. 俯视图：同一个 BEV 世界 + 6 个相机在 ego 坐标系中的位置、朝向和近似视锥。
#      这里 x 轴是车头方向，y 轴是车左方向。
fig, ax = plt.subplots(figsize=(7, 7))

# road and lanes in the shared world
ax.fill_between([-22, 30], -5.2, 5.2, color="0.88", alpha=0.8)
for y_lane in [-3.5, 0.0, 3.5]:
    ax.plot([-22, 30], [y_lane, y_lane], color="white", linewidth=2, linestyle="--" if y_lane == 0 else "-")

# ego vehicle rectangle
car_x = torch.tensor([-2.0, 2.0, 2.0, -2.0, -2.0])
car_y = torch.tensor([-0.9, -0.9, 0.9, 0.9, -0.9])
ax.plot(car_x, car_y, color="black", linewidth=2)
ax.arrow(0, 0, 1.4, 0, width=0.025, head_width=0.18, head_length=0.25, color="black")
ax.text(1.55, 0, "ego front", va="center")

# shared objects
for obj in world_objects:
    cx_obj, cy_obj = obj["xy"]
    length, width, _ = obj["size"]
    rect_x = [cx_obj - length/2, cx_obj + length/2, cx_obj + length/2, cx_obj - length/2, cx_obj - length/2]
    rect_y = [cy_obj - width/2, cy_obj - width/2, cy_obj + width/2, cy_obj + width/2, cy_obj - width/2]
    ax.fill(rect_x, rect_y, color=obj["color"], alpha=0.75)
    ax.text(cx_obj, cy_obj, obj["name"], fontsize=7, ha="center", va="center")
for cone_x, cone_y in world_cones:
    ax.scatter(cone_x, cone_y, color="orange", marker="^", s=45)

# cameras and rough FOV
ray_depth = 16.0
corner_pixels = torch.tensor([
    [0.0, cy, 1.0],
    [imW - 1.0, cy, 1.0],
])
for cam_i, name in enumerate(camera_names):
    pos = trans[0, cam_i]
    rot = rots[0, cam_i]
    forward = rot.matmul(torch.tensor([0.0, 0.0, 1.0]))
    rays_cam = torch.inverse(intrins[0, cam_i]).matmul(corner_pixels.T).T * ray_depth
    rays_ego = rays_cam.matmul(rot.T) + pos

    ax.scatter(pos[0], pos[1], s=45)
    ax.arrow(
        float(pos[0]), float(pos[1]),
        float(forward[0]) * 2.2, float(forward[1]) * 2.2,
        width=0.02, head_width=0.20, head_length=0.25,
        length_includes_head=True,
    )
    ax.plot([pos[0], rays_ego[0, 0]], [pos[1], rays_ego[0, 1]], linestyle="--", alpha=0.55)
    ax.plot([pos[0], rays_ego[1, 0]], [pos[1], rays_ego[1, 1]], linestyle="--", alpha=0.55)
    ax.text(float(pos[0]) + 0.12, float(pos[1]) + 0.12, name, fontsize=8)

ax.set_title("00b shared BEV scene and camera rig")
ax.set_xlabel("ego x: forward meters")
ax.set_ylabel("ego y: left meters")
ax.set_xlim(-18, 30)
ax.set_ylim(-18, 18)
ax.set_aspect("equal", adjustable="box")
ax.grid(True)
_display_fig(fig)


In [ ]:
# 00c. 一致性检查：后续所有步骤都会继续使用这些变量。
print("downstream variables")
print("imgs       ", tuple(imgs.shape))
print("rots       ", tuple(rots.shape))
print("trans      ", tuple(trans.shape))
print("intrins    ", tuple(intrins.shape))
print("post_rots  ", tuple(post_rots.shape), "max |I - post_rots| =", (post_rots - torch.eye(3).view(1, 1, 3, 3)).abs().max().item())
print("post_trans ", tuple(post_trans.shape), "max |post_trans| =", post_trans.abs().max().item())
print()
print("Important: 第 7 步及之后不会重新创建这些输入；它们会沿着源码一步步往下流动。")


## 7. Geometry：frustum -> ego 3D


In [ ]:
# 01 frustum template
B, N, _ = trans.shape
points = model.frustum
show_points3d("01 frustum template", points)


In [ ]:
# 02 undo post translation (identity in this example)
points = model.frustum - post_trans.view(B, N, 1, 1, 1, 3)
show_points3d("02 undo post translation (identity in this example)", points)


In [ ]:
# 03 inverse post_rots (identity in this example)
post_rots_inv = torch.inverse(post_rots)
show_hist("03 inverse post_rots (identity in this example)", post_rots_inv)


In [ ]:
# 04 apply inverse post_rots (identity in this example)
points = post_rots_inv.view(B, N, 1, 1, 1, 3, 3).matmul(points.unsqueeze(-1)).squeeze(-1)
show_points3d("04 apply inverse post_rots (identity in this example)", points)


In [ ]:
# 05 pixel times depth
points = torch.cat((points[:, :, :, :, :, :2] * points[:, :, :, :, :, 2:3], points[:, :, :, :, :, 2:3]), 5)
show_points3d("05 pixel times depth", points)


In [ ]:
# 06 inverse intrinsics
intrins_inv = torch.inverse(intrins)
show_hist("06 inverse intrinsics", intrins_inv)


In [ ]:
# 07 combine rots and inverse intrinsics
combine = rots.matmul(intrins_inv)
show_hist("07 combine rots and inverse intrinsics", combine)


In [ ]:
# 08 camera coordinates to ego rotation
points = combine.view(B, N, 1, 1, 1, 3, 3).matmul(points.unsqueeze(-1)).squeeze(-1)
show_points3d("08 camera coordinates to ego rotation", points)


In [ ]:
# 09 add ego translation
geom = points + trans.view(B, N, 1, 1, 1, 3)
show_points3d("09 add ego translation", geom)


## 8. Camera Lift：image -> frustum feature


In [ ]:
# 10 merge batch and camera
x = imgs.view(B * N, C, imH, imW)
show_heatmap("10 merge batch and camera", x)


In [ ]:
# 11 efficientnet stem conv
cam = model.camencode
endpoints = {}
x = cam.trunk._conv_stem(x)
show_heatmap("11 efficientnet stem conv", x)


In [ ]:
# 12 efficientnet stem batchnorm
x = cam.trunk._bn0(x)
show_heatmap("12 efficientnet stem batchnorm", x)


In [ ]:
# 13 efficientnet stem swish
x = cam.trunk._swish(x)
prev_x = x
show_heatmap("13 efficientnet stem swish", x)


In [ ]:
# 14 efficientnet MBConv block 0
drop_connect_rate = cam.trunk._global_params.drop_connect_rate
if drop_connect_rate:
    drop_connect_rate *= float(0) / len(cam.trunk._blocks)
x = cam.trunk._blocks[0](x, drop_connect_rate=drop_connect_rate)
show_heatmap("14 efficientnet MBConv block 0", x)
if prev_x.size(2) > x.size(2):
    endpoint_name = "reduction_{}".format(len(endpoints) + 1)
    endpoints[endpoint_name] = prev_x
    print("saved", endpoint_name, _tensor_stats(prev_x))
prev_x = x


In [ ]:
# 15 efficientnet MBConv block 1
drop_connect_rate = cam.trunk._global_params.drop_connect_rate
if drop_connect_rate:
    drop_connect_rate *= float(1) / len(cam.trunk._blocks)
x = cam.trunk._blocks[1](x, drop_connect_rate=drop_connect_rate)
show_heatmap("15 efficientnet MBConv block 1", x)
if prev_x.size(2) > x.size(2):
    endpoint_name = "reduction_{}".format(len(endpoints) + 1)
    endpoints[endpoint_name] = prev_x
    print("saved", endpoint_name, _tensor_stats(prev_x))
prev_x = x


In [ ]:
# 16 efficientnet MBConv block 2
drop_connect_rate = cam.trunk._global_params.drop_connect_rate
if drop_connect_rate:
    drop_connect_rate *= float(2) / len(cam.trunk._blocks)
x = cam.trunk._blocks[2](x, drop_connect_rate=drop_connect_rate)
show_heatmap("16 efficientnet MBConv block 2", x)
if prev_x.size(2) > x.size(2):
    endpoint_name = "reduction_{}".format(len(endpoints) + 1)
    endpoints[endpoint_name] = prev_x
    print("saved", endpoint_name, _tensor_stats(prev_x))
prev_x = x


In [ ]:
# 17 efficientnet MBConv block 3
drop_connect_rate = cam.trunk._global_params.drop_connect_rate
if drop_connect_rate:
    drop_connect_rate *= float(3) / len(cam.trunk._blocks)
x = cam.trunk._blocks[3](x, drop_connect_rate=drop_connect_rate)
show_heatmap("17 efficientnet MBConv block 3", x)
if prev_x.size(2) > x.size(2):
    endpoint_name = "reduction_{}".format(len(endpoints) + 1)
    endpoints[endpoint_name] = prev_x
    print("saved", endpoint_name, _tensor_stats(prev_x))
prev_x = x


In [ ]:
# 18 efficientnet MBConv block 4
drop_connect_rate = cam.trunk._global_params.drop_connect_rate
if drop_connect_rate:
    drop_connect_rate *= float(4) / len(cam.trunk._blocks)
x = cam.trunk._blocks[4](x, drop_connect_rate=drop_connect_rate)
show_heatmap("18 efficientnet MBConv block 4", x)
if prev_x.size(2) > x.size(2):
    endpoint_name = "reduction_{}".format(len(endpoints) + 1)
    endpoints[endpoint_name] = prev_x
    print("saved", endpoint_name, _tensor_stats(prev_x))
prev_x = x


In [ ]:
# 19 efficientnet MBConv block 5
drop_connect_rate = cam.trunk._global_params.drop_connect_rate
if drop_connect_rate:
    drop_connect_rate *= float(5) / len(cam.trunk._blocks)
x = cam.trunk._blocks[5](x, drop_connect_rate=drop_connect_rate)
show_heatmap("19 efficientnet MBConv block 5", x)
if prev_x.size(2) > x.size(2):
    endpoint_name = "reduction_{}".format(len(endpoints) + 1)
    endpoints[endpoint_name] = prev_x
    print("saved", endpoint_name, _tensor_stats(prev_x))
prev_x = x


In [ ]:
# 20 efficientnet MBConv block 6
drop_connect_rate = cam.trunk._global_params.drop_connect_rate
if drop_connect_rate:
    drop_connect_rate *= float(6) / len(cam.trunk._blocks)
x = cam.trunk._blocks[6](x, drop_connect_rate=drop_connect_rate)
show_heatmap("20 efficientnet MBConv block 6", x)
if prev_x.size(2) > x.size(2):
    endpoint_name = "reduction_{}".format(len(endpoints) + 1)
    endpoints[endpoint_name] = prev_x
    print("saved", endpoint_name, _tensor_stats(prev_x))
prev_x = x


In [ ]:
# 21 efficientnet MBConv block 7
drop_connect_rate = cam.trunk._global_params.drop_connect_rate
if drop_connect_rate:
    drop_connect_rate *= float(7) / len(cam.trunk._blocks)
x = cam.trunk._blocks[7](x, drop_connect_rate=drop_connect_rate)
show_heatmap("21 efficientnet MBConv block 7", x)
if prev_x.size(2) > x.size(2):
    endpoint_name = "reduction_{}".format(len(endpoints) + 1)
    endpoints[endpoint_name] = prev_x
    print("saved", endpoint_name, _tensor_stats(prev_x))
prev_x = x


In [ ]:
# 22 efficientnet MBConv block 8
drop_connect_rate = cam.trunk._global_params.drop_connect_rate
if drop_connect_rate:
    drop_connect_rate *= float(8) / len(cam.trunk._blocks)
x = cam.trunk._blocks[8](x, drop_connect_rate=drop_connect_rate)
show_heatmap("22 efficientnet MBConv block 8", x)
if prev_x.size(2) > x.size(2):
    endpoint_name = "reduction_{}".format(len(endpoints) + 1)
    endpoints[endpoint_name] = prev_x
    print("saved", endpoint_name, _tensor_stats(prev_x))
prev_x = x


In [ ]:
# 23 efficientnet MBConv block 9
drop_connect_rate = cam.trunk._global_params.drop_connect_rate
if drop_connect_rate:
    drop_connect_rate *= float(9) / len(cam.trunk._blocks)
x = cam.trunk._blocks[9](x, drop_connect_rate=drop_connect_rate)
show_heatmap("23 efficientnet MBConv block 9", x)
if prev_x.size(2) > x.size(2):
    endpoint_name = "reduction_{}".format(len(endpoints) + 1)
    endpoints[endpoint_name] = prev_x
    print("saved", endpoint_name, _tensor_stats(prev_x))
prev_x = x


In [ ]:
# 24 efficientnet MBConv block 10
drop_connect_rate = cam.trunk._global_params.drop_connect_rate
if drop_connect_rate:
    drop_connect_rate *= float(10) / len(cam.trunk._blocks)
x = cam.trunk._blocks[10](x, drop_connect_rate=drop_connect_rate)
show_heatmap("24 efficientnet MBConv block 10", x)
if prev_x.size(2) > x.size(2):
    endpoint_name = "reduction_{}".format(len(endpoints) + 1)
    endpoints[endpoint_name] = prev_x
    print("saved", endpoint_name, _tensor_stats(prev_x))
prev_x = x


In [ ]:
# 25 efficientnet MBConv block 11
drop_connect_rate = cam.trunk._global_params.drop_connect_rate
if drop_connect_rate:
    drop_connect_rate *= float(11) / len(cam.trunk._blocks)
x = cam.trunk._blocks[11](x, drop_connect_rate=drop_connect_rate)
show_heatmap("25 efficientnet MBConv block 11", x)
if prev_x.size(2) > x.size(2):
    endpoint_name = "reduction_{}".format(len(endpoints) + 1)
    endpoints[endpoint_name] = prev_x
    print("saved", endpoint_name, _tensor_stats(prev_x))
prev_x = x


In [ ]:
# 26 efficientnet MBConv block 12
drop_connect_rate = cam.trunk._global_params.drop_connect_rate
if drop_connect_rate:
    drop_connect_rate *= float(12) / len(cam.trunk._blocks)
x = cam.trunk._blocks[12](x, drop_connect_rate=drop_connect_rate)
show_heatmap("26 efficientnet MBConv block 12", x)
if prev_x.size(2) > x.size(2):
    endpoint_name = "reduction_{}".format(len(endpoints) + 1)
    endpoints[endpoint_name] = prev_x
    print("saved", endpoint_name, _tensor_stats(prev_x))
prev_x = x


In [ ]:
# 27 efficientnet MBConv block 13
drop_connect_rate = cam.trunk._global_params.drop_connect_rate
if drop_connect_rate:
    drop_connect_rate *= float(13) / len(cam.trunk._blocks)
x = cam.trunk._blocks[13](x, drop_connect_rate=drop_connect_rate)
show_heatmap("27 efficientnet MBConv block 13", x)
if prev_x.size(2) > x.size(2):
    endpoint_name = "reduction_{}".format(len(endpoints) + 1)
    endpoints[endpoint_name] = prev_x
    print("saved", endpoint_name, _tensor_stats(prev_x))
prev_x = x


In [ ]:
# 28 efficientnet MBConv block 14
drop_connect_rate = cam.trunk._global_params.drop_connect_rate
if drop_connect_rate:
    drop_connect_rate *= float(14) / len(cam.trunk._blocks)
x = cam.trunk._blocks[14](x, drop_connect_rate=drop_connect_rate)
show_heatmap("28 efficientnet MBConv block 14", x)
if prev_x.size(2) > x.size(2):
    endpoint_name = "reduction_{}".format(len(endpoints) + 1)
    endpoints[endpoint_name] = prev_x
    print("saved", endpoint_name, _tensor_stats(prev_x))
prev_x = x


In [ ]:
# 29 efficientnet MBConv block 15
drop_connect_rate = cam.trunk._global_params.drop_connect_rate
if drop_connect_rate:
    drop_connect_rate *= float(15) / len(cam.trunk._blocks)
x = cam.trunk._blocks[15](x, drop_connect_rate=drop_connect_rate)
show_heatmap("29 efficientnet MBConv block 15", x)
if prev_x.size(2) > x.size(2):
    endpoint_name = "reduction_{}".format(len(endpoints) + 1)
    endpoints[endpoint_name] = prev_x
    print("saved", endpoint_name, _tensor_stats(prev_x))
prev_x = x


In [ ]:
# 30 save final efficientnet endpoint
endpoint_name = "reduction_{}".format(len(endpoints) + 1)
endpoints[endpoint_name] = x
show_heatmap("30 save final efficientnet endpoint", x)


In [ ]:
# 31 cam up1 upsample reduction_5
x = cam.up1.up(endpoints["reduction_5"])
show_heatmap("31 cam up1 upsample reduction_5", x)


In [ ]:
# 32 cam up1 concat reduction_4 skip
x = torch.cat([endpoints["reduction_4"], x], dim=1)
show_heatmap("32 cam up1 concat reduction_4 skip", x)


In [ ]:
# 33 cam up1 layer 0
x = cam.up1.conv[0](x)
show_heatmap("33 cam up1 layer 0: " + cam.up1.conv[0].__class__.__name__, x)


In [ ]:
# 34 cam up1 layer 1
x = cam.up1.conv[1](x)
show_heatmap("34 cam up1 layer 1: " + cam.up1.conv[1].__class__.__name__, x)


In [ ]:
# 35 cam up1 layer 2
x = cam.up1.conv[2](x)
show_heatmap("35 cam up1 layer 2: " + cam.up1.conv[2].__class__.__name__, x)


In [ ]:
# 36 cam up1 layer 3
x = cam.up1.conv[3](x)
show_heatmap("36 cam up1 layer 3: " + cam.up1.conv[3].__class__.__name__, x)


In [ ]:
# 37 cam up1 layer 4
x = cam.up1.conv[4](x)
show_heatmap("37 cam up1 layer 4: " + cam.up1.conv[4].__class__.__name__, x)


In [ ]:
# 38 cam up1 layer 5
x = cam.up1.conv[5](x)
show_heatmap("38 cam up1 layer 5: " + cam.up1.conv[5].__class__.__name__, x)


In [ ]:
# 39 depthnet 1x1 conv
x = cam.depthnet(x)
show_heatmap("39 depthnet 1x1 conv", x)


In [ ]:
# 40 depth logits slice
depth_logits = x[:, :model.D]
show_heatmap("40 depth logits slice", depth_logits)


In [ ]:
# 41 depth softmax distribution
depth = cam.get_depth_dist(depth_logits)
show_heatmap("41 depth softmax distribution", depth)


In [ ]:
# 42 image feature slice
img_feat = x[:, model.D:(model.D + model.camC)]
show_heatmap("42 image feature slice", img_feat)


In [ ]:
# 43 depth unsqueeze
depth_for_lift = depth.unsqueeze(1)
show_heatmap("43 depth unsqueeze", depth_for_lift)


In [ ]:
# 44 image feature unsqueeze
img_feat_for_lift = img_feat.unsqueeze(2)
show_heatmap("44 image feature unsqueeze", img_feat_for_lift)


In [ ]:
# 45 lift depth times image feature
cam_x = depth_for_lift * img_feat_for_lift
show_heatmap("45 lift depth times image feature", cam_x)


In [ ]:
# 46 restore B and N dims
cam_x = cam_x.view(B, N, model.camC, model.D, imH // model.downsample, imW // model.downsample)
show_heatmap("46 restore B and N dims", cam_x)


In [ ]:
# 47 permute to geometry order
cam_feats = cam_x.permute(0, 1, 3, 4, 5, 2)
show_heatmap("47 permute to geometry order", cam_feats)


## 9. Splat：frustum feature -> BEV grid


In [ ]:
# 48 compute flattened point count
B, N, D, fH, fW, featC = cam_feats.shape
Nprime = B * N * D * fH * fW
print("Nprime:", Nprime)


In [ ]:
# 49 flatten frustum features
splat_x = cam_feats.reshape(Nprime, featC)
show_heatmap("49 flatten frustum features", splat_x)


In [ ]:
# 50 ego xyz to voxel ijk
geom_vox = ((geom - (model.bx - model.dx / 2.0)) / model.dx).long()
show_points3d("50 ego xyz to voxel ijk", geom_vox.float())


In [ ]:
# 51 flatten voxel indices
geom_vox = geom_vox.view(Nprime, 3)
show_points2d("51 flatten voxel indices", geom_vox.float())


In [ ]:
# 52 create batch index
batch_ix = torch.cat([torch.full([Nprime // B, 1], ix, device=splat_x.device, dtype=torch.long) for ix in range(B)])
show_hist("52 create batch index", batch_ix.float())


In [ ]:
# 53 append batch index
geom_vox = torch.cat((geom_vox, batch_ix), 1)
show_points2d("53 append batch index", geom_vox.float())


In [ ]:
# 54 in-bounds mask
kept = ((geom_vox[:, 0] >= 0) & (geom_vox[:, 0] < model.nx[0]) & (geom_vox[:, 1] >= 0) & (geom_vox[:, 1] < model.nx[1]) & (geom_vox[:, 2] >= 0) & (geom_vox[:, 2] < model.nx[2]))
show_mask("54 in-bounds mask", kept)


In [ ]:
# 55 keep in-bounds features
splat_x = splat_x[kept]
show_heatmap("55 keep in-bounds features", splat_x)


In [ ]:
# 56 keep in-bounds voxel indices
geom_vox = geom_vox[kept]
show_points2d("56 keep in-bounds voxel indices", geom_vox.float())


In [ ]:
# 57 encode voxel rank
ranks = (geom_vox[:, 0] * (model.nx[1] * model.nx[2] * B) + geom_vox[:, 1] * (model.nx[2] * B) + geom_vox[:, 2] * B + geom_vox[:, 3])
show_hist("57 encode voxel rank", ranks.float())


In [ ]:
# 58 argsort ranks
sorts = ranks.argsort()
show_hist("58 argsort ranks", sorts.float())


In [ ]:
# 59 sort features by rank
splat_x = splat_x[sorts]
show_heatmap("59 sort features by rank", splat_x)


In [ ]:
# 60 sort voxel indices by rank
geom_vox = geom_vox[sorts]
show_points2d("60 sort voxel indices by rank", geom_vox.float())


In [ ]:
# 61 sort ranks
ranks = ranks[sorts]
show_hist("61 sort ranks", ranks.float())


In [ ]:
# 62 cumsum sorted features
cumsum_x = splat_x.cumsum(0)
show_heatmap("62 cumsum sorted features", cumsum_x)


In [ ]:
# 63 voxel-end mask
voxel_end = torch.ones(cumsum_x.shape[0], device=cumsum_x.device, dtype=torch.bool)
voxel_end[:-1] = ranks[1:] != ranks[:-1]
show_mask("63 voxel-end mask", voxel_end)


In [ ]:
# 64 keep voxel cumsum endpoints
voxel_x = cumsum_x[voxel_end]
show_heatmap("64 keep voxel cumsum endpoints", voxel_x)


In [ ]:
# 65 keep voxel endpoint indices
voxel_geom = geom_vox[voxel_end]
show_points2d("65 keep voxel endpoint indices", voxel_geom.float())


In [ ]:
# 66 subtract adjacent cumsums
voxel_x = torch.cat((voxel_x[:1], voxel_x[1:] - voxel_x[:-1]))
show_heatmap("66 subtract adjacent cumsums", voxel_x)


In [ ]:
# 67 allocate BEV grid
bev_grid = torch.zeros((B, featC, model.nx[2], model.nx[0], model.nx[1]), device=voxel_x.device)
show_heatmap("67 allocate BEV grid", bev_grid)


In [ ]:
# 68 scatter into BEV grid
bev_grid[voxel_geom[:, 3], :, voxel_geom[:, 2], voxel_geom[:, 0], voxel_geom[:, 1]] = voxel_x
show_heatmap("68 scatter into BEV grid", bev_grid)


In [ ]:
# 69 collapse Z into channels
bev_x = torch.cat(bev_grid.unbind(dim=2), 1)
show_heatmap("69 collapse Z into channels", bev_x)


## 10. BEV Encode：BEV feature -> logits


In [ ]:
# 70 BEV encoder input
bev = model.bevencode
x = bev_x
show_heatmap("70 BEV encoder input", x)


In [ ]:
# 71 BEV conv1
x = bev.conv1(x)
show_heatmap("71 BEV conv1", x)


In [ ]:
# 72 BEV batchnorm1
x = bev.bn1(x)
show_heatmap("72 BEV batchnorm1", x)


In [ ]:
# 73 BEV relu1
x = bev.relu(x)
show_heatmap("73 BEV relu1", x)


In [ ]:
# 74 BEV resnet layer1
x1 = bev.layer1(x)
show_heatmap("74 BEV resnet layer1", x1)


In [ ]:
# 75 BEV resnet layer2
x = bev.layer2(x1)
show_heatmap("75 BEV resnet layer2", x)


In [ ]:
# 76 BEV resnet layer3
x = bev.layer3(x)
show_heatmap("76 BEV resnet layer3", x)


In [ ]:
# 77 BEV up1 upsample
x = bev.up1.up(x)
show_heatmap("77 BEV up1 upsample", x)


In [ ]:
# 78 BEV up1 concat skip
x = torch.cat([x1, x], dim=1)
show_heatmap("78 BEV up1 concat skip", x)


In [ ]:
# 79 BEV up1 layer 0
x = bev.up1.conv[0](x)
show_heatmap("79 BEV up1 layer 0: " + bev.up1.conv[0].__class__.__name__, x)


In [ ]:
# 80 BEV up1 layer 1
x = bev.up1.conv[1](x)
show_heatmap("80 BEV up1 layer 1: " + bev.up1.conv[1].__class__.__name__, x)


In [ ]:
# 81 BEV up1 layer 2
x = bev.up1.conv[2](x)
show_heatmap("81 BEV up1 layer 2: " + bev.up1.conv[2].__class__.__name__, x)


In [ ]:
# 82 BEV up1 layer 3
x = bev.up1.conv[3](x)
show_heatmap("82 BEV up1 layer 3: " + bev.up1.conv[3].__class__.__name__, x)


In [ ]:
# 83 BEV up1 layer 4
x = bev.up1.conv[4](x)
show_heatmap("83 BEV up1 layer 4: " + bev.up1.conv[4].__class__.__name__, x)


In [ ]:
# 84 BEV up1 layer 5
x = bev.up1.conv[5](x)
show_heatmap("84 BEV up1 layer 5: " + bev.up1.conv[5].__class__.__name__, x)


In [ ]:
# 85 BEV up2 layer 0
x = bev.up2[0](x)
show_heatmap("85 BEV up2 layer 0: " + bev.up2[0].__class__.__name__, x)


In [ ]:
# 86 BEV up2 layer 1
x = bev.up2[1](x)
show_heatmap("86 BEV up2 layer 1: " + bev.up2[1].__class__.__name__, x)


In [ ]:
# 87 BEV up2 layer 2
x = bev.up2[2](x)
show_heatmap("87 BEV up2 layer 2: " + bev.up2[2].__class__.__name__, x)


In [ ]:
# 88 BEV up2 layer 3
x = bev.up2[3](x)
show_heatmap("88 BEV up2 layer 3: " + bev.up2[3].__class__.__name__, x)


In [ ]:
# 89 BEV up2 layer 4
x = bev.up2[4](x)
show_heatmap("89 BEV up2 layer 4: " + bev.up2[4].__class__.__name__, x)


In [ ]:
# 90 final logits
logits = x
show_heatmap("90 final BEV segmentation logits", logits)
print("final logits shape:", tuple(logits.shape))


## 11. 和原始 `model.forward()` 对齐检查


In [ ]:
with torch.no_grad():
    direct_logits = model(imgs, rots, trans, intrins, post_rots, post_trans)

print("manual logits shape:", tuple(logits.shape))
print("direct logits shape:", tuple(direct_logits.shape))
print("max abs diff:", (logits - direct_logits).abs().max().item())
